# Refined geodesics on the synthetic t-SNE embeddings

Qualitative companion to `synthetic_geodesics.ipynb`: five Stage III
**refined + projected** geodesics ("ours") per scenario, taken from the
`geodesics-full-s42` run, overlaid on the per-scenario t-SNE panels of
`synthetic_data_gallery.ipynb`.

## How the overlay is built

* **Background** — t-SNE of the tree steps plus capped subsamples of the clean
  `cell_means` and the noisy counts (perplexity 30, PCA init, seed 42): the
  gallery protocol, so the panels match `synthetic_data_gallery.ipynb`.
* **Curves** — a Stage III path is a 31-waypoint polyline in the 50-D PCA
  working space (shown: `paths_proj`, i.e. the refinement projected back onto
  the clean manifold, including the endpoints). t-SNE has no out-of-sample
  transform, so each waypoint is placed at a distance-weighted average of the
  2-D positions of its 15 nearest neighbours among the embedded points
  (bandwidth = distance to the 15th neighbour). This is a display-only local
  projection: every paper metric is computed in the 50-D space.
* **Pair choice** — every scenario must cover a pre-declared set of categories
  (trunk, leaves, junctions). Within each category the pair pool is
  ranked by the sum of the per-pair off-tree distance
  (`mean__refine_proj`) and the relative length error
  `|L_ref_proj - L_tree| / L_tree`; the best pair is taken first, and any
  further pairs are chosen to be far from the already selected curves on the
  manifold.
* **Markers** — filled circle = curve start, cross = curve end, i.e. the
  projected `x0`/`x1`; the curve colours are numbered per panel and match the
  table printed below.


## 0. Setup

In [ ]:
from __future__ import annotations

import sys
import time
from pathlib import Path

from IPython.display import Image, Markdown, display


def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, here.parent, here.parent.parent):
        if (candidate / "experiments" / "synthetic_data").is_dir():
            return candidate
    raise RuntimeError(
        "Could not locate the repository root (expected experiments/synthetic_data/)."
    )


REPO_ROOT = _find_repo_root()

for _p in (REPO_ROOT, REPO_ROOT / "src"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib import patheffects
from matplotlib.lines import Line2D
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors

from experiments.synthetic_data.common.clean import load_clean_tree
from notebooks.helpers.scenarios import SCENARIO_META
from notebooks.helpers.style import DPI, apply_paper_style, savefig


def show_fig(path: Path) -> None:
    """Display the PNG sibling inline (savefig writes both PNG and SVG)."""
    print("saved:", path)
    display(Image(filename=str(path.with_suffix(".png"))))


apply_paper_style()
print(f"REPO_ROOT  = {REPO_ROOT}")
print(f"Figure DPI = {DPI}")


## 1. Geometry artifacts and Stage III geodesics

In [ ]:
DATA_ROOT = REPO_ROOT / "experiments" / "synthetic_data" / "synthetic" / "data"
RESULTS_ROOT = REPO_ROOT / "results" / "synthetic_data"
OUT_DIR = REPO_ROOT / "notebooks" / "figures" / "sc_geodesics"

SCENARIOS = ["linear", "bifurcation", "tree"]
SEED = 42
RUN_ID = "geodesics-full-s42"  # must be the seed-42 run: pairs + t-SNE share the seed
N_SHOW = 5
TSNE_MAX_CELLS = 6000
TSNE_PERPLEXITY = 30.0
KNN_K = 15

OUT_DIR.mkdir(parents=True, exist_ok=True)


def load_scenario(scenario: str) -> dict:
    """Geometry artifacts of one scenario (same fields as the gallery notebook)."""
    geo = DATA_ROOT / scenario / "geometry"
    return {
        "scenario": scenario,
        "tree": load_clean_tree(geo / "clean_tree.npz"),
        "labels": pd.read_csv(geo / "labels_aligned.csv"),
        "X_noisy": np.load(geo / "X_pca.npy"),
        "X_means": np.load(geo / "X_cell_means_pca.npy"),
    }


def load_payload(scenario: str) -> dict:
    """Stage III geodesics payload (paths, lengths, pair bookkeeping)."""
    return torch.load(
        RESULTS_ROOT / scenario / RUN_ID / "geodesics" / "paths.pt",
        map_location="cpu",
        weights_only=False,
    )


DATA = {scenario: load_scenario(scenario) for scenario in SCENARIOS}
PAYLOADS = {scenario: load_payload(scenario) for scenario in SCENARIOS}

for scenario in SCENARIOS:
    payload = PAYLOADS[scenario]
    print(
        f"{scenario:12s} {len(payload['L_tree']):5d} pairs · "
        f"{len(DATA[scenario]['tree'].path_names)} paths · "
        f"paths_proj {tuple(payload['paths_proj'].shape)}"
    )


## 2. Pick five well-performing pairs per scenario

In [ ]:
#: Required categories per scenario: trunk and leaf branches plus every
#: junction type. ``cross:`` categories are ``<path>-<segment>`` pairs and
#: ``same:`` categories cover one branch end to end.
PAIR_QUOTAS: dict[str, dict[str, int]] = {
    "linear": {
        "same:Path1": 2,
        "same:Path2": 2,
        "cross:Path1-late|Path2-early": 1,
    },
    "bifurcation": {
        "same:Path1": 1,
        "same:Path2": 1,
        "cross:Path1-late|Path2-mid": 1,
        "cross:Path1-late|Path3-mid": 1,
        "cross:Path2-mid|Path3-mid": 1,
    },
    "tree": {
        "same:Path1": 1,
        "same:Path5": 1,
        "cross:Path1-late|Path4-mid": 1,
        "cross:Path4-mid|Path5-mid": 1,
        "cross:Path6-mid|Path7-mid": 1,
    },
}


def _rank(values: np.ndarray) -> np.ndarray:
    """0-based ranks of ``values`` (ascending)."""
    return np.argsort(np.argsort(values)).astype(float)


def select_pairs(
    payload: dict,
    off_gt: dict,
    quotas: dict[str, int],
    *,
    top: int = 40,
    floor_q: float = 0.02,
) -> tuple[list[int], dict[str, np.ndarray]]:
    """Pick per-category pairs that work well and are spread over the manifold.

    Pairs are scored by ``rank(off-tree distance) + rank(relative length
    error)``; the relative error uses a quantile floor on ``L_tree`` so that
    near-zero junction pairs do not dominate. After the first pick of a
    category, the candidate whose 50-D chord midpoint is farthest from the
    already selected midpoints wins.
    """
    categories = np.asarray(payload["category"])
    lengths = np.asarray(payload["L_tree"], dtype=float)
    lengths_ref = np.asarray(payload["L_ref_proj"], dtype=float)
    off = np.asarray(off_gt["mean__refine_proj"], dtype=float)
    floor = max(float(np.quantile(lengths, floor_q)), 1e-6)
    rel_err = np.abs(lengths_ref - lengths) / np.maximum(lengths, floor)
    midpoints = 0.5 * (np.asarray(payload["x0"]) + np.asarray(payload["x1"]))

    chosen: list[int] = []
    anchors: list[np.ndarray] = []
    for category, want in quotas.items():
        pool = np.flatnonzero(categories == category)
        score = _rank(off[pool]) + _rank(rel_err[pool])
        order = pool[np.argsort(score)][:top]
        while want > 0 and order.size:
            if anchors:
                spread = np.min(
                    [np.linalg.norm(midpoints[order] - a, axis=1) for a in anchors],
                    axis=0,
                )
                pick = int(order[np.argmax(spread)])
            else:
                pick = int(order[0])
            order = order[order != pick]
            chosen.append(pick)
            anchors.append(midpoints[pick])
            want -= 1
    return chosen, {"off": off, "rel_err": rel_err}


SELECTIONS: dict[str, list[int]] = {}
METRICS: dict[str, dict[str, np.ndarray]] = {}
for scenario in SCENARIOS:
    with np.load(
        RESULTS_ROOT / scenario / RUN_ID / "geodesics" / "off_gt.npz"
    ) as off_gt:
        chosen, metrics = select_pairs(
            PAYLOADS[scenario], off_gt, PAIR_QUOTAS[scenario]
        )
    assert len(chosen) == N_SHOW, (scenario, chosen)
    SELECTIONS[scenario] = chosen
    METRICS[scenario] = metrics
    cats = np.asarray(PAYLOADS[scenario]["category"])
    print(scenario)
    for position, pair in enumerate(chosen, start=1):
        print(
            f"  {position} · pair {pair:4d} · {cats[pair]:30s} "
            f"L_tree {PAYLOADS[scenario]['L_tree'][pair]:.4f} · "
            f"off-tree {metrics['off'][pair]:.4f} · "
            f"rel. length err {metrics['rel_err'][pair]:.3f}"
        )


## 3. t-SNE embedding and waypoint placement

In [ ]:
def _subsample(n: int, cap: int, rng: np.random.Generator) -> np.ndarray:
    """Gallery subsampling: capped, sorted indices."""
    if cap <= 0 or cap >= n:
        return np.arange(n)
    return np.sort(rng.choice(n, size=cap, replace=False))


def build_embedding(entry: dict) -> dict:
    """Gallery t-SNE of the tree steps, clean cells and noisy cells."""
    rng = np.random.default_rng(SEED)
    idx_means = _subsample(len(entry["X_means"]), TSNE_MAX_CELLS, rng)
    idx_noisy = _subsample(len(entry["X_noisy"]), TSNE_MAX_CELLS, rng)
    X = np.vstack(
        [entry["tree"].points, entry["X_means"][idx_means], entry["X_noisy"][idx_noisy]]
    )
    tick = time.time()
    Y = TSNE(
        n_components=2,
        perplexity=TSNE_PERPLEXITY,
        init="pca",
        random_state=SEED,
        n_jobs=-1,
    ).fit_transform(X)
    print(
        f"    {entry['scenario']}: t-SNE on {len(X)} points in {time.time() - tick:.0f}s"
    )
    n_tree, n_means = len(entry["tree"].points), len(idx_means)
    return {
        "tree": Y[:n_tree],
        "means": Y[n_tree : n_tree + n_means],
        "noisy": Y[n_tree + n_means :],
        "X_base": X,
        "Y_base": Y,
    }


def project_curves(embedding: dict, paths: np.ndarray, k: int = KNN_K) -> np.ndarray:
    """Place 50-D waypoints via a soft kNN average of embedded neighbours."""
    neighbours = NearestNeighbors(n_neighbors=k).fit(embedding["X_base"])
    flat = paths.reshape(-1, paths.shape[-1])
    distances, indices = neighbours.kneighbors(flat)
    bandwidth = distances[:, -1:].clip(min=1e-12)
    weights = np.exp(-0.5 * (distances / bandwidth) ** 2)
    weights /= weights.sum(axis=1, keepdims=True)
    projected = np.einsum("nk,nkd->nd", weights, embedding["Y_base"][indices])
    return projected.reshape(paths.shape[0], paths.shape[1], 2)


COORDS: dict[str, dict] = {}
CURVES: dict[str, np.ndarray] = {}
for scenario in SCENARIOS:
    payload = PAYLOADS[scenario]
    chosen = SELECTIONS[scenario]
    COORDS[scenario] = build_embedding(DATA[scenario])
    CURVES[scenario] = project_curves(
        COORDS[scenario], payload["paths_proj"].numpy()[chosen]
    )
    print(
        f"    {scenario}: {len(chosen)} curves × {CURVES[scenario].shape[1]} "
        f"waypoints projected onto the embedding"
    )


## 4. Figure

In [ ]:
def pair_label(category: str) -> str:
    """Short legend label for a ``same:``/``cross:`` pair category."""
    if category.startswith("same:"):
        return f"{category[5:]} (same)"
    left, right = category[6:].split("|")
    return f"{left.rsplit('-', 1)[0]}→{right.rsplit('-', 1)[0]}"


def draw_panel(
    ax,
    entry: dict,
    coords: dict,
    curves: np.ndarray,
    labels: list[str],
    *,
    show_ylabel: bool = False,
) -> None:
    """One scenario panel: grey background, tree skeleton, coloured curves."""
    ax.scatter(
        coords["noisy"][:, 0],
        coords["noisy"][:, 1],
        s=1.0,
        c="0.78",
        alpha=0.4,
        linewidths=0,
        rasterized=True,
        zorder=1,
    )
    ax.scatter(
        coords["means"][:, 0],
        coords["means"][:, 1],
        s=2.0,
        c="0.5",
        alpha=0.35,
        linewidths=0,
        rasterized=True,
        zorder=2,
    )
    tree = entry["tree"]
    for path in range(len(tree.path_names)):
        start, count = int(tree.point_start[path]), int(tree.n_points[path])
        points = coords["tree"][start : start + count]
        ax.plot(
            points[:, 0],
            points[:, 1],
            color="white",
            lw=2.6,
            zorder=3,
            solid_capstyle="round",
        )
        ax.plot(
            points[:, 0],
            points[:, 1],
            color="0.3",
            lw=1.2,
            zorder=4,
            solid_capstyle="round",
        )

    cmap = mpl.colormaps["Dark2"]
    for position, curve in enumerate(curves):
        color = cmap(position)
        ax.plot(
            curve[:, 0],
            curve[:, 1],
            color=color,
            lw=2.2,
            zorder=6,
            solid_capstyle="round",
            solid_joinstyle="round",
            path_effects=[
                patheffects.withStroke(linewidth=4.8, foreground="white", alpha=0.95)
            ],
        )
        ax.scatter(
            curve[0, 0],
            curve[0, 1],
            color=color,
            s=48,
            marker="o",
            edgecolor="white",
            linewidths=1.1,
            zorder=7,
        )
        ax.scatter(
            curve[-1, 0],
            curve[-1, 1],
            color=color,
            s=62,
            marker="X",
            edgecolor="white",
            linewidths=1.1,
            zorder=7,
        )

    meta = SCENARIO_META[entry["scenario"]]
    ax.set_title(f"{meta['label']}\n{meta['description']}", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("t-SNE 1", labelpad=4)
    if show_ylabel:
        ax.set_ylabel("t-SNE 2", labelpad=4)

    handles = [
        Line2D([], [], color=cmap(position), lw=2.2, solid_capstyle="round")
        for position in range(len(curves))
    ]
    handles += [
        Line2D([], [], color="0.3", marker="o", ls="none", markersize=5, label="start"),
        Line2D([], [], color="0.3", marker="X", ls="none", markersize=6, label="end"),
    ]
    ax.legend(
        handles,
        labels + ["start", "end"],
        loc="upper center",
        bbox_to_anchor=(0.5, -0.16),
        ncol=2,
        frameon=False,
        fontsize=7.6,
        handlelength=1.5,
        columnspacing=0.8,
        borderaxespad=0.0,
    )


fig, axes = plt.subplots(1, len(SCENARIOS), figsize=(13.8, 5.2))
ROWS: list[dict] = []
for ax, scenario in zip(axes, SCENARIOS):
    cats = np.asarray(PAYLOADS[scenario]["category"])
    chosen = SELECTIONS[scenario]
    labels = [
        f"{position} · {pair_label(str(cats[pair]))}"
        for position, pair in enumerate(chosen, start=1)
    ]
    draw_panel(
        ax,
        DATA[scenario],
        COORDS[scenario],
        CURVES[scenario],
        labels,
        show_ylabel=ax is axes[0],
    )
    for position, pair in enumerate(chosen, start=1):
        payload, metrics = PAYLOADS[scenario], METRICS[scenario]
        ROWS.append(
            {
                "scenario": scenario,
                "n": position,
                "pair": pair,
                "category": str(cats[pair]),
                "L_tree": float(payload["L_tree"][pair]),
                "L_ref_proj": float(payload["L_ref_proj"][pair]),
                "rel_len_err": float(metrics["rel_err"][pair]),
                "off_gt": float(metrics["off"][pair]),
            }
        )

fig.suptitle(
    "Refined + projected geodesics in the synthetic t-SNE embeddings\n"
    "five well-performing pairs per scenario (low off-tree distance, small "
    "length error)",
    fontsize=11,
    y=1.04,
)
fig.subplots_adjust(wspace=0.08)
FIGURE_PATH = savefig(fig, OUT_DIR / "03_tsne_overlay")
show_fig(FIGURE_PATH)


## 5. Selected pairs

In [ ]:
SELECTION = pd.DataFrame(ROWS)
LINES = [
    "| scenario | # | pair | category | $L_\\mathrm{tree}$ | "
    "$L_\\mathrm{ref,proj}$ | rel. length err. | off-tree dist. |",
    "|---|---:|---:|---|---:|---:|---:|---:|",
]
for row in SELECTION.itertuples(index=False):
    LINES.append(
        f"| {row.scenario} | {row.n} | {row.pair} | `{row.category}` | "
        f"{row.L_tree:.4f} | {row.L_ref_proj:.4f} | "
        f"{row.rel_len_err:.3f} | {row.off_gt:.4f} |"
    )
display(Markdown("\n".join(LINES)))
